# Analise Exploratoria - Desafio RMI (Educacao)

Objetivo: entender a estrutura e a **qualidade** dos dados brutos para justificar
as decisoes de modelagem no dbt (camadas staging -> intermediate -> marts).

Usamos **DuckDB** para ler os Parquet direto (sem BigQuery), o que torna a EDA
rapida e reproduzivel localmente.

> Os dados estao no bucket publico `gs://case_vagas/rmi` (arquivos sem extensao).
> Configure `DATA_DIR` abaixo para a pasta com os arquivos baixados, ou use a
> leitura via HTTP (celula seguinte).


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect()

# Opcao A: arquivos ja baixados localmente (ajuste o caminho).
DATA_DIR = "../data"

# Opcao B (sem download): ler direto do GCS via httpfs.
#   con.execute("INSTALL httpfs; LOAD httpfs;")
#   DATA_DIR = "https://storage.googleapis.com/case_vagas/rmi"

def tbl(nome):
    return f"read_parquet('{DATA_DIR}/{nome}')"

print("DuckDB", duckdb.__version__)

## 1. Volumetria e grao de cada tabela

In [ ]:
resumo = []
for nome in ["aluno", "escola", "turma", "avaliacao", "frequencia"]:
    n = con.execute(f"SELECT count(*) FROM {tbl(nome)}").fetchall()[0][0]
    cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {tbl(nome)}").fetchall()]
    resumo.append({"tabela": nome, "linhas": n, "colunas": ", ".join(cols)})
pd.DataFrame(resumo)

## 2. Achado-chave: existem DUAS frequencias, e so uma tem sinal

O desafio sugere analisar "absenteismo cronico" (frequencia < 75%). A tabela
`frequencia` (por periodo) parece o caminho obvio, mas vamos ver que ela nao
tem variancia (97-100%). A frequencia COM sinal esta em `avaliacao.frequencia`
(geral por bimestre, 0-100%), onde ~10,4% dos alunos ficam abaixo de 75%.

In [ ]:
freq_stats = con.execute(f'''
    SELECT min(frequencia) AS minimo, max(frequencia) AS maximo,
           avg(frequencia) AS media,
           count(*) FILTER (WHERE frequencia < 75)  AS abaixo_75,
           count(*) FILTER (WHERE frequencia < 90)  AS abaixo_90,
           count(*) AS total
    FROM {tbl("frequencia")}
''').df()
freq_stats

In [ ]:
dist = con.execute(f'''
    SELECT round(frequencia, 1) AS faixa, count(*) AS n
    FROM {tbl("frequencia")}
    GROUP BY 1 ORDER BY 1
''').df()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dist["faixa"].astype(str), dist["n"], color="#2563eb")
ax.set_title("Distribuicao do percentual de frequencia (toda a rede)")
ax.set_xlabel("Percentual de frequencia")
ax.set_ylabel("Registros")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print("Conclusao: nenhum registro < 97%. O mart de absenteismo < 75% retorna zero.")

## 3. Achado: `avaliacao` e WIDE e tem colunas problematicas

- Formato wide (uma coluna por disciplina) -> precisa de UNPIVOT no staging.
- `disciplina_3` (Ingles) e **100% nula** (0 notas preenchidas).
- Ja a coluna `frequencia` da avaliacao esta ~99% preenchida (so 1.734 nulas).

In [ ]:
con.execute(f'''
    SELECT
        count(*) AS linhas,
        count(disciplina_1) AS notas_portugues,
        count(disciplina_2) AS notas_ciencias,
        count(disciplina_3) AS notas_ingles,
        count(disciplina_4) AS notas_matematica,
        count(frequencia)   AS freq_preenchida
    FROM {tbl("avaliacao")}
''').df()

## 4. Achado: integridade referencial

- `id_aluno` concilia 100% entre as tabelas -> chave de join confiavel.
- `frequencia.id_turma` tem ~12% de orfaos vs matricula -> NAO usar como join.

In [ ]:
checks = {}
checks["freq_aluno_orfao"] = con.execute(f'''
    SELECT count(*) FROM {tbl("frequencia")} f
    WHERE NOT EXISTS (SELECT 1 FROM {tbl("aluno")} a WHERE a.id_aluno = f.id_aluno)
''').fetchall()[0][0]
checks["freq_turma_orfao"] = con.execute(f'''
    SELECT count(*) FROM {tbl("frequencia")} f
    WHERE NOT EXISTS (SELECT 1 FROM {tbl("turma")} t WHERE t.id_turma = f.id_turma)
''').fetchall()[0][0]
checks["freq_total"] = con.execute(f"SELECT count(*) FROM {tbl('frequencia')}").fetchall()[0][0]
pd.Series(checks)

## 5. Duplicatas de grao

Tanto `frequencia` quanto `avaliacao` tem linhas duplicadas no grao natural,
tratadas com `row_number()` no staging.

In [ ]:
dup_freq = con.execute(f'''
    SELECT count(*) AS total,
           count(*) - count(DISTINCT (id_aluno, id_turma, id_escola, data_inicio, data_fim, disciplina)) AS duplicatas
    FROM {tbl("frequencia")}
''').df()
dup_aval = con.execute(f'''
    SELECT count(*) AS total,
           count(*) - count(DISTINCT (id_aluno, id_turma, bimestre)) AS duplicatas
    FROM {tbl("avaliacao")}
''').df()
print("frequencia:"); print(dup_freq.to_string(index=False))
print("avaliacao:");  print(dup_aval.to_string(index=False))

## 6. Sintese -> decisoes de modelagem

| Achado | Decisao no dbt |
|---|---|
| `id_aluno` como BYTES | `to_hex()` no staging para chave STRING estavel |
| Frequencia sempre >= 97% | Mart de absenteismo correto mas com taxa 0; expor distribuicao |
| `avaliacao` wide + Ingles nulo | UNPIVOT no staging; teste documenta a ausencia |
| `frequencia.id_turma` orfao (~12%) | Join por `id_aluno`; relationship de turma como `warn` |
| Duplicatas de grao | Dedup com `row_number()` no staging |
| `turma` = matricula (sem serie/turno) | Renomear para matriculas; mart de turma usa tamanho |
